# PySpark Introductorio: pandas vs PySpark
## Proyecto P4 - Pipeline ETL Airbnb Buenos Aires

### Objetivo
Comparar pandas y PySpark para el mismo análisis sobre el dataset de Airbnb Buenos Aires
Entender cuándo tiene sentido usar cada herramienta.

### Dataset
- listings_clean.csv: 27.348 propiedades x 72 columnas
- Tamaño: ~47MB (pequeño, ideal para pandas)

## 1. ¿Qué es Apache Spark?

Apache Spark es un motor de procesamiento distribuido diseñado para analizar
grandes volúmenes de datos que no caben en la memoria de una sola máquina.

**Cuándo usar pandas:**
- Dataset < 2GB
- Análisis exploratorio y prototipado
- Una sola máquina

**Cuándo usar PySpark:**
- Dataset > 10GB (gigabytes a terabytes)
- Procesamiento en Cluster de servidores
- Pipelines de producción con datos masivos

**Nuestro caso:** 47MB -> pandas es suficiente
PySpark se usa aquí con fines didácticos

In [1]:
## Imports y configuración
import pandas as pd
import time
from pathlib import Path

DATA_RAW = Path("../data/raw")

## 2. Análisis con pandas

In [2]:
# Cargar datos con pandas
inicio = time.time()

df_pandas = pd.read_csv(DATA_RAW / "listings_clean.csv")

tiempo_carga_pandas = time.time() -  inicio
print(f"Tiempo de carga: {tiempo_carga_pandas:.2f} segundos")
print(f"Shape: {df_pandas.shape}")

Tiempo de carga: 0.78 segundos
Shape: (27348, 72)


In [3]:
# Análisis con pandas: top 10 barrios por precio promedio
inicio = time.time()

resultado_pandas = (
    df_pandas
    .groupby("neighbourhood_cleansed")
    .agg(
        precio_promedio=("review_scores_rating","mean"),
        total_propiedades=("id","count")
    )
    .sort_values("total_propiedades", ascending=False)
    .head(10)
)

tiempo_analisis_pandas = time.time() - inicio
print(f"Tiempo de análisis: {tiempo_analisis_pandas:.4f} segundos")
resultado_pandas

Tiempo de análisis: 0.0117 segundos


,precio_promedio,total_propiedades
neighbourhood_cleansed,,
Palermo,4.796268,8963
Recoleta,4.788486,3828
San Nicolas,4.760551,2156
Belgrano,4.772440,1318
Retiro,4.769965,1310
Monserrat,4.710436,1256
Balvanera,4.709015,1051
Almagro,4.757405,1011
Villa Crespo,4.786946,927


## 3. Análisis con PySpark

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Inicializar Spark
inicio = time.time()

spark = SparkSession.builder \
    .appName("Airbnb Analytics") \
    .master("local[*]") \
    .getOrCreate()
    
tiempo_inicio_spark = time.time() - inicio
print(f"Tiempo de inicio Spark: {tiempo_inicio_spark:.2f} segundos")

Tiempo de inicio Spark: 1.95 segundos


In [5]:
# Cargar datos con PySpark
inicio = time.time()

df_spark = spark.read.csv(
    str(DATA_RAW / "listings_clean.csv"),
    header=True,
    inferSchema=True
)

tiempo_carga_spark = time.time() - inicio
print(f"Tiempo de carga: {tiempo_carga_spark:.2f} segundos")
print(f"Filas: {df_spark.count()}")
print(f"Columnas: {len(df_spark.columns)}")

Tiempo de carga: 3.82 segundos
Filas: 27348
Columnas: 72


In [ ]:
# Análisis con PySpark: top 10 barrios por propiedades
inicio = time.time()

resultado_spark = (
    df_spark
    .groupby("neighbourhood_cleansed")
    .agg(
        F.round(F.avg("review_scores_rating"),2).alias("precio_promedio"),
        F.count("id").alias("total_propiedades")
    )
    .orderBy(F.col("total_propiedades").desc())
    .limit(10)
)

resultado_spark.show()
tiempo_analisis_spark = time.time() - inicio
print(f"Tiempo de análisis: {tiempo_analisis_spark:.4f} segundos")

## 4. Comparativa pandas vs PySpark

In [7]:
print("=" * 50)
print("COMPARATIVA PANDAS vs PYSPARK")
print("=" * 50)
print(f"\nDataset: 27.348 filas × 72 columnas (47MB)\n")
print(f"{'Métrica':<25} {'pandas':>10} {'PySpark':>10}")
print("-" * 50)
print(f"{'Inicio sesión':<25} {'0.00s':>10} {tiempo_inicio_spark:>9.2f}s")
print(f"{'Carga de datos':<25} {tiempo_carga_pandas:>9.2f}s {tiempo_carga_spark:>9.2f}s")
print(f"{'Análisis groupby':<25} {tiempo_analisis_pandas:>9.4f}s {tiempo_analisis_spark:>9.4f}s")
print(f"\n{'CONCLUSIÓN':}")
print(f"Para 47MB → pandas es 5x más rápido y más simple")
print(f"PySpark tiene sentido a partir de datasets > 10GB")
print(f"o cuando se necesita procesamiento distribuido en cluster")

COMPARATIVA PANDAS vs PYSPARK

Dataset: 27.348 filas × 72 columnas (47MB)

Métrica                       pandas    PySpark
--------------------------------------------------
Inicio sesión                  0.00s      1.95s
Carga de datos                 0.78s      3.82s
Análisis groupby             0.0117s    1.0548s

CONCLUSIÓN
Para 47MB → pandas es 5x más rápido y más simple
PySpark tiene sentido a partir de datasets > 10GB
o cuando se necesita procesamiento distribuido en cluster


## 5. Conclusiones

### ¿Cuándo usar pandas?
- Dataset cabe en RAM de tu máquina (< 2GB)
- Análisis exploratorio y prototipado rápido
- Proyectos individuales o equipos pequeños
- Sintaxis más simple y directa

### ¿Cuándo usar PySpark?
- Dataset no cabe en RAM (> 10GB, terabytes)
- Procesamiento distribuido en cluster de servidores
- Pipelines de producción con datos masivos
- Streaming de datos en tiempo real
- Integración con ecosistema Big Data (HDFS, Kafka, Delta Lake)

### Resultado de este análisis (47MB)
| Métrica | pandas | PySpark |
|---------|--------|---------|
| Inicio sesión | 0.00s | 1.95s |
| Carga de datos | 0.78s | 3.82s |
| Análisis groupby | 0.01s | 1.05s |

**Conclusión:** Para datasets pequeños como este (47MB), pandas es 
más rápido y más simple. PySpark agrega overhead sin beneficio real.
En producción con terabytes de datos, PySpark es la herramienta correcta.